In [49]:
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage

from langchain_anthropic import ChatAnthropic
import operator

In [50]:
llm = ChatAnthropic(
    model="claude-haiku-4-5"
)

In [51]:
class ChatState(TypedDict):
    messages: Annotated[list, operator.add]

In [52]:
def chatbot(state: ChatState):
    response = llm.invoke(state["messages"])

    return {
        "messages": [
            AIMessage(content=response.content)
        ]
    }

In [53]:
builder = StateGraph(ChatState)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

memory = InMemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [54]:
config = {
    "configurable": {
        "thread_id": "user-1"
    }
}

result = graph.invoke(
    {
         "messages": [
            HumanMessage(content="give me a random number?")
        ]
    },
    config=config
)

print(result["messages"][-1].content)

42


In [55]:
state = graph.get_state(config)

print(state)

StateSnapshot(values={'messages': [HumanMessage(content='give me a random number?', additional_kwargs={}, response_metadata={}), AIMessage(content='42', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}, next=(), config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f16fe7a-7565-61c5-8001-93bd933265a4'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-06-24T16:13:36.077649+00:00', parent_config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f16fe7a-6aff-65e2-8000-946fa2d33263'}}, tasks=(), interrupts=())


In [56]:
result = graph.invoke(
   {
         "messages": [
            HumanMessage(content="add 10 to the result ")
        ]
    },
    config=config
)

print(result["messages"][-1].content)

42 + 10 = 52


In [57]:
graph.get_state(config).values

{'messages': [HumanMessage(content='give me a random number?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='42', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='add 10 to the result ', additional_kwargs={}, response_metadata={}),
  AIMessage(content='42 + 10 = 52', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [58]:
graph.get_state( {
    "configurable": {
        "thread_id": "user-1",
        "checkpoint_id": "1f16fe7a-7565-61c5-8001-93bd933265a4"
    }
})

StateSnapshot(values={'messages': [HumanMessage(content='give me a random number?', additional_kwargs={}, response_metadata={}), AIMessage(content='42', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}, next=(), config={'configurable': {'thread_id': 'user-1', 'checkpoint_id': '1f16fe7a-7565-61c5-8001-93bd933265a4'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-06-24T16:13:36.077649+00:00', parent_config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f16fe7a-6aff-65e2-8000-946fa2d33263'}}, tasks=(), interrupts=())

In [ ]:
result = graph.invoke(
   
    None
    ,
    config={
        "configurable": {
            "thread_id": "user-1",
            "checkpoint_id": "1f16fe7a-7565-61c5-8001-93bd933265a4"
        }
    }
)

print(result["messages"][-1].content)

42
